In [66]:
import pandas as pd
from pathlib import Path
import numpy as np


sachs_path = Path(
    "/vol/bitbucket/ad6013/Research/CausalInferenceNeuralProcess/CITNP/datasets/synth_training_data/sachs/sachs_data_cleaned.csv"
)

In [67]:
def extract_obs_samples(sachs_data: pd.DataFrame, num_samples: int):
    """
    Extract observational samples from the Sachs dataset.
    """
    obs_samples = sachs_data[sachs_data["intervention_name"] == "obs"]
    data_columns = obs_samples.iloc[:, :11].to_numpy()
    return data_columns[:num_samples]


def extract_int_samples(sachs_data: pd.DataFrame, intvn: str):
    """
    Extract interventional samples from the Sachs dataset.
    """
    int_samples = sachs_data[sachs_data["intervention_name"] == intvn]
    data_columns = int_samples.iloc[:, :11].to_numpy()
    return data_columns

In [68]:
all_datasets = []

sachs = pd.read_csv(sachs_path)

# Extract 750 samples of obs data
sachs_obs = extract_obs_samples(sachs, 750)

# We need a differnt dataset for each interventional data
unique_int = sachs["intervention_name"].unique()
for intvn in unique_int:
    if intvn == "obs":
        continue
    final_dataset = {}

    final_dataset["obs_data"] = sachs_obs[None]

    sachs_int = extract_int_samples(sachs, intvn)

    final_dataset["int_data"] = sachs_int[None]

    # Find the intervention index
    intvn_index = sachs.columns.get_loc(intvn)
    intvn_index = np.array([intvn_index])[None]

    final_dataset["intvn_indices"] = intvn_index

    variable_counts = np.array([sachs.shape[1]])[None]
    final_dataset["variable_counts"] = variable_counts

    causal_graph = np.zeros((sachs.shape[1], sachs.shape[1]))[None]
    final_dataset["causal_graphs"] = causal_graph

    all_datasets.append(final_dataset)

In [69]:
import h5py


save_path = Path(
    "/vol/bitbucket/ad6013/Research/CausalInferenceNeuralProcess/CITNP/datasets/synth_training_data/sachs/test"
)
save_path.mkdir(parents=True, exist_ok=True)


for idx, dataset in enumerate(all_datasets):
    with h5py.File(f"{save_path}/data_idx{idx}.hdf5", "w") as f:
        for key, value in dataset.items():
            if value is None:
                continue
            f.create_dataset(key, data=value)


In [53]:
intvn_index

0

In [57]:
sachs.columns.get_loc('PIP2')

3

In [46]:
sachs_obs

array([[3.27336401, 2.58021683, 2.17702187, ..., 2.83321334, 3.80443779,
        3.68887945],
       [3.5807373 , 2.80336038, 2.50959926, ..., 1.21491274, 2.80336038,
        4.11903717],
       [4.08429423, 3.78645978, 2.68102153, ..., 2.43361336, 3.46260601,
        2.97041447],
       ...,
       [4.01096295, 2.6461748 , 3.34638915, ..., 2.67414865, 3.29212629,
        2.22137504],
       [3.62434093, 2.79728133, 2.92316158, ..., 2.68102153, 3.58905912,
        3.19458313],
       [3.74242022, 3.13113691, 3.28466357, ..., 3.3638416 , 3.65065824,
        3.04927304]], shape=(750, 11))

In [17]:
sachs

,praf,pmek,plcg,PIP2,PIP3,p44/42,pakts473,PKA,PKC,P38,pjnk,intervention_name
0,3.273364,2.580217,2.177022,2.906901,4.074142,1.888584,2.833213,6.025866,2.833213,3.804438,3.688879,obs
1,3.580737,2.803360,2.509599,2.821379,2.095561,2.923162,3.481240,5.863631,1.214913,2.803360,4.119037,obs
2,4.084294,3.786460,2.681022,2.322388,2.564949,2.701361,3.481240,5.998937,2.433613,3.462606,2.970414,obs
3,4.290459,4.416428,3.139833,2.602690,0.254642,1.763017,2.468100,6.269096,2.617396,3.353407,3.139833,obs
4,3.517498,2.985682,1.646734,2.275214,3.210844,3.049273,3.830813,5.720312,1.539015,3.246491,4.398146,obs
...,...,...,...,...,...,...,...,...,...,...,...,...
5841,2.024193,1.591274,2.149434,2.933857,4.762174,2.370244,2.850707,6.152733,2.660260,3.058707,2.862201,PIP2
5842,3.417727,3.624341,1.465568,2.803360,3.173878,1.474763,2.701361,5.521461,1.178655,0.157004,1.870263,PIP2
5843,2.068128,3.077312,2.078191,5.105945,3.830813,3.113515,3.605498,6.079933,3.005683,3.039749,2.131797,PIP2
5844,1.969906,2.791165,1.026042,2.442347,2.442347,2.995732,3.328627,6.144186,3.443618,4.262680,3.202746,PIP2


In [18]:
sachs_obs = sachs[sachs["intervention_name"] == "obs"]

In [25]:
sachs_obs_data_cols = sachs_obs.iloc[:, :11]

In [26]:
sachs_obs_data_numpy = sachs_obs_data_cols.to_numpy()

In [28]:
# Take the first 750 samples
sachs_obs_data_numpy_samples = sachs_obs_data_numpy[:750]

In [ ]:
all_interventions = sachs["intervention_name"].unique()
intervention_dict = {}
for intervention in all_interventions:
    if intervention == "obs":
        continue
    interventions = sachs[sachs["intervention_name"] == intervention]
    interventions_datacols = interventions.iloc[:, :11].to_numpy()
    intervention_dict[intervention] = interventions_datacols


In [35]:
[v.shape[0] for k,v in intervention_dict.items()]

[910, 723, 810, 799, 848]

In [ ]:
import requests
import pandas as pd


url = "https://www.bnlearn.com/book-crc/code/sachs.data.txt.gz"
# You can choose the name and path for the downloaded file
file_name = "sachs.data.txt.gz"

print(f"Downloading {file_name} from {url}...")

# Send a GET request to the URL
response = requests.get(url, stream=True)

# Raise an exception if the request was unsuccessful (e.g., 404 Not Found)
response.raise_for_status()

# Open the local file in write-binary mode
with open(file_name, 'wb') as f:
    # Iterate over the response data in chunks and write to file
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

print(f"File '{file_name}' downloaded successfully.")

File 'sachs.data.txt.gz' downloaded successfully.


In [10]:
df['INT'].unique()

array([8, 0, 7, 9, 4, 2])

In [1]:
import cdt
data, graph = cdt.data.load_dataset('sachs')

Detecting 1 CUDA device(s).


In [3]:
data

,praf,pmek,plcg,PIP2,PIP3,p44/42,pakts473,PKA,PKC,P38,pjnk
0,26.4,13.20,8.82,18.30,58.80,6.61,17.0,414.0,17.00,44.90,40.00
1,35.9,16.50,12.30,16.80,8.13,18.60,32.5,352.0,3.37,16.50,61.50
2,59.4,44.10,14.60,10.20,13.00,14.90,32.5,403.0,11.40,31.90,19.50
3,73.0,82.80,23.10,13.50,1.29,5.83,11.8,528.0,13.70,28.60,23.10
4,33.7,19.80,5.19,9.73,24.80,21.10,46.1,305.0,4.66,25.70,81.30
...,...,...,...,...,...,...,...,...,...,...,...
7461,49.1,12.40,32.80,27.90,22.70,11.70,38.2,1144.0,1.00,2.55,1.00
7462,23.3,4.61,17.80,22.10,14.90,48.70,67.3,922.0,1.00,9.82,1.00
7463,28.1,4.49,18.80,20.20,10.20,3.08,21.9,730.0,1.00,1.75,2.00
7464,34.6,7.10,5.73,20.70,15.10,32.20,41.4,813.0,44.50,1382.00,2.44
